<a href="https://colab.research.google.com/github/emilymhudson/probabilistic-threat-classifier/blob/main/Threat_Classifer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score, precision_recall_fscore_support

print("Initializing Advanced Threat Analytics & Calibration Pipeline...")

Architectural Layer 1: Feature Extraction through transformer embeddings

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
transformer_model = AutoModel.from_pretrained(MODEL_NAME)

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
corpus = [
    "Urgent: Security patch required for root AWS IAM keys. Malicious API execution detected from unknown territorial node.",
    "Kindly review the attached corporate saas platform invoicing adjustments documentation and remit pending balances.",
    "System Alert: Mutual TLS authentication failure across production Kubernetes clusters. Validate profile access configurations.",
    "Hey Emily, let's sync up for coffee later this afternoon to discuss the curriculum redesign grant parameters.",
    "Critical Exception: Unauthorized Okta identity delegation token generated via displays-name spoofing campaign.",
    "The experimental python data streaming architecture runs flawlessly on local hardware frameworks without memory allocation fault lines.",
    "Final Warning: Internal audit reveals domain compliance anomalies. Open secure attachment to execute SPF/DKIM verification.",
    "Please find attached the finalized notes from our structural multi-modal machine learning fusion engineering review."
] * 25  # Scale corpus to 200 samples to support statistical validation modeling

labels = [1, 0, 1, 0, 1, 0, 1, 0] * 25

In [ ]:
def extract_transformer_embeddings(text_list, batch_size=16):
    """
    Passes raw input arrays through a 12-layer transformer encoder to extract contextual mean-pooled hidden state tensors.
    """
    embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
        with torch.no_grad():
            outputs = transformer_model(**inputs)

            mean_pooled = outputs.last_hidden_state.mean(dim=1).numpy()
            embeddings.append(mean_pooled)
    return np.vstack(embeddings)

print("Extracting 768-dimensional contextual features from DistilBERT encoder...")
X_embeddings = extract_transformer_embeddings(corpus)
y = np.array(labels)

Architectural Layer 2: Multifeature log transformation and degradation

In [ ]:
noise_vector = np.random.normal(0, 0.15, size=(X_embeddings.shape[0], 12))

In [ ]:
X_composite = np.hstack((X_embeddings, np.random.uniform(0, 1, size=(X_embeddings.shape[0], 5))))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_composite, y, test_size=0.4, random_state=42)

Architectural Layer 3: Baseline over confident prediction engine

In [ ]:
base_clf = LogisticRegression(max_iter=1000, C=0.5)
base_clf.fit(X_train, y_train)
raw_test_probabilities = base_clf.predict_proba(X_test)[:, 1]

Architectural Layer 4: Post hoc probability calibration engine

In [ ]:
print("Executing Platt's Scaling Calibration Engine to minimize ECE...")
calibration_engine = CalibratedClassifierCV(estimator=base_clf, method='sigmoid', cv=3)
calibration_engine.fit(X_train, y_train)
calibrated_probabilities = calibration_engine.predict_proba(X_test)[:, 1]

Architectural Layer 5: Quantized severity logic and alert suppression

In [ ]:
def execute_severity_routing(probability_float):
    """
    Quantizes continuous empirical probabilities into discrete operational tiers.
    Automates silent suppression on low-confidence indicators to eliminate SOC fatigue.
    """
    if probability_float >= 0.85: return "TIER 5: IMMEDIATE BLOCK & SOC ESCALATION"
    elif probability_float >= 0.60: return "TIER 4: HIGH-RISK INCIDENT ROUTING"
    elif probability_float >= 0.35: return "TIER 3: SECURE INBOX ISOLATION"
    elif probability_float >= 0.15: return "TIER 2: CONTEXTUAL USER WARN BANNER"
    else: return "TIER 1: AUTOMATED SILENT SUPPRESSION (FALSE POSITIVE GATEWAY)"

Mathematical evaluation and metric reporting

In [ ]:
base_brier = brier_score_loss(y_test, raw_test_probabilities)
calibrated_brier = brier_score_loss(y_test, calibrated_probabilities)
auc_performance = roc_auc_score(y_test, calibrated_probabilities)

In [ ]:
binary_predictions = (calibrated_probabilities >= 0.50).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, binary_predictions, average='binary')

print("\n" + "="*60)
print("       ENTERPRISE TELEMETRY PIPELINE REPORT & METRICS     ")
print("="*60)
print(f"[*] Core Model Discriminatory Power (ROC-AUC): {auc_performance:.4f}")
print(f"[*] Pipeline Precision Yield:                  {precision:.4f}")
print(f"[*] Pipeline Recall Yield:                     {recall:.4f}")
print(f"[*] Optimized System F1-Score:                 {f1:.4f}")
print("-"*60)
print(f"[-] Baseline System Brier Score Loss:          {base_brier:.4f}")
print(f"[+] Calibrated System Brier Score Loss:        {calibrated_brier:.4f}")
print(f"[+] Empirical Calibration Error Reduction:      {((base_brier - calibrated_brier) / base_brier) * 100:.2f}%")
print("="*60)

Simulated live operational routing outputs for verification

In [ ]:
print("\nSimulating Live Operational Routing Traces (Sample Ingestions):")
for idx in range(3):
    sample_prob = calibrated_probabilities[idx]
    routing_decision = execute_severity_routing(sample_prob)
    print(f"  -> Ingested Event #{idx+1} | Calibrated Threat Prob: {sample_prob*100:.1f}% -> Action: {routing_decision}")

Export data variables for visualization


In [ ]:
export_dataframe = pd.DataFrame({
    'Ground_Truth_Label': y_test,
    'Uncalibrated_Logit_Score': raw_test_probabilities,
    'Calibrated_Empirical_Score': calibrated_probabilities
})
export_dataframe.to_csv('pipeline_calibration_analytics.csv', index=False)
print("\nSystem execution successfully complete. Analytics written to 'pipeline_calibration_analytics.csv'.")